# Smart Farming: Multi-Model Server with Cloudflare Worker Gateway

| Service | Port | With a GPU | Without a GPU (CPU only) |
|---|---|---|---|
| Vision models (`main:app`) | 8001 | fast | slower |
| IndicTrans2 translation | 8002 | fast | about 1-3 s per sentence |
| Qwen3 advisory (`/advise`) | 8003 | Qwen3-4B, 4-bit | Qwen3 GGUF through llama.cpp (1.7B by default, 4B optional) |

Each service gets its own Cloudflare tunnel. All URLs change on every restart.
Cloudflare quick tunnels cut a request after about 100 seconds (HTTP 524), so keep CPU answers short
and rely on the advisory cache.

### Cell 0: Hardware check

In [ ]:
import subprocess, os
res = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True)
if res.returncode == 0:
    print("GPU:", res.stdout.strip(), "-> Qwen3-4B will run in 4-bit on the GPU.")
else:
    print(f"No GPU. CPU cores: {os.cpu_count()}. All three services will run on CPU.")
    print("Qwen will use a small GGUF model through llama.cpp. Expect roughly 20-60 s per new advisory.")

### Cell 1: Mount Drive and sync vision service and knowledge base


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/Smart-Farming
!git clone --depth 1 --filter=blob:none --sparse https://github.com/manthan2876/Smart-Farming.git /content/Smart-Farming
%cd /content/Smart-Farming
!git sparse-checkout set model_service data

SRC = '/content/Smart-Farming/model_service'
DST = '/content/drive/MyDrive/SmartFarming'
!rm -rf "{DST}"
!cp -r "{SRC}" "{DST}"

# Ensure advisory knowledge base is in Drive
!cp -f /content/Smart-Farming/data/advisory_kb.json "{DST}/advisory_kb.json" 2>/dev/null || true
!cp -f "{SRC}/advisory_kb.json" "{DST}/advisory_kb.json" 2>/dev/null || true

print("Vision service and Knowledge Base synced to Drive.")

import os, json
_kb = DST + "/advisory_kb.json"
if os.path.exists(_kb):
    print("advisory_kb.json entries:", len(json.load(open(_kb, encoding="utf-8"))))
else:
    print("WARNING: advisory_kb.json not found in SmartFarming/. /advise will answer 'knowledge not found' "
          "for every crop until you add it (key format: 'Crop|Disease', e.g. 'Tomato|Late Blight').")

### Cell 2: Install dependencies (Vision + IndicTrans2 + Qwen3-4B bitsandbytes)


In [ ]:
!pip install -q -r /content/drive/MyDrive/SmartFarming/requirements.txt

# transformers must stay on 4.x for IndicTrans2; Qwen3 needs at least 4.51
!pip install -q "transformers>=4.51,<5" sentencepiece sacremoses
!pip install -q git+https://github.com/VarunGumma/IndicTransToolkit

# GPU path for Qwen: 4-bit quantization
!pip install -q bitsandbytes accelerate

# CPU path for Qwen: llama.cpp. Takes a prebuilt CPU wheel when one exists for this Python;
# otherwise pip builds it from source, which can take 5-10 minutes.
!pip install -q "llama-cpp-python>=0.3.12" --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu || pip install -q "llama-cpp-python>=0.3.12"

# Create directories for services
!mkdir -p /content/drive/MyDrive/IndicTransServer
!mkdir -p /content/drive/MyDrive/AdvisoryServer

# Sanity check
!python -c "import transformers, huggingface_hub; print('transformers:', transformers.__version__, '| hub:', huggingface_hub.__version__); from IndicTransToolkit.processor import IndicProcessor; print('Toolkit import OK!')"
!python -c "import llama_cpp; print('llama_cpp', llama_cpp.__version__, '- CPU path for Qwen is ready')"

### Cell 3: Download cloudflared


In [ ]:
#@title Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!/content/cloudflared --version


### Cell 4: Write the IndicTrans2 server file to Drive


In [ ]:
%%writefile /content/drive/MyDrive/IndicTransServer/translate_server.py
import os, json, threading
os.environ.setdefault("HF_HOME", "/content/hf_cache")

import torch
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

LANGS = {"hi": "hin_Deva", "gu": "guj_Gujr"}
MODEL_NAME = "ai4bharat/indictrans2-en-indic-dist-200M"
CACHE_FILE = "/content/drive/MyDrive/indictrans_cache.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_BEAMS = 4 if DEVICE == "cuda" else 2
CHUNK = 16 if DEVICE == "cuda" else 8
if DEVICE == "cpu":
    torch.set_num_threads(os.cpu_count() or 2)
print(f"[Init] Device: {DEVICE}")

print("[Init] Loading IndicTrans2 tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, trust_remote_code=True).to(DEVICE).eval()
processor = IndicProcessor(inference=True)
lock = threading.Lock()

cache = {}
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, encoding="utf-8") as f:
            cache = json.load(f)
        print(f"[Cache] Loaded {len(cache)} cached translations from Drive.")
    except Exception as e:
        print(f"[Cache] Could not load cache: {e}")

def save_cache():
    try:
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False)
    except Exception:
        pass

def run(texts, tgt):
    batch = processor.preprocess_batch(texts, src_lang="eng_Latn", tgt_lang=tgt)
    inputs = tokenizer(batch, padding=True, truncation=True, max_length=256,
                       return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = model.generate(**inputs, num_beams=NUM_BEAMS, max_length=256, use_cache=False)
    return processor.postprocess_batch(
        tokenizer.batch_decode(out, skip_special_tokens=True), lang=tgt)

run(["Field health looks good."], "hin_Deva")
print("[Init] IndicTrans2 warmed up and ready.")

app = FastAPI(title="IndicTrans2 Translation Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

class TranslationRequest(BaseModel):
    texts: list[str]
    target: str

@app.get("/health")
def health():
    return {"status": "healthy", "service": "indictrans2", "device": DEVICE}

@app.post("/translate")
def translate(req: TranslationRequest):
    lang = req.target.lower()
    tgt = LANGS.get(lang)
    if not tgt:
        return {"translations": req.texts, "error": f"Unsupported target {req.target}"}

    def key(t):
        return f"{lang}:{t}"

    with lock:
        todo = [t for t in dict.fromkeys(req.texts) if key(t) not in cache]
        for i in range(0, len(todo), CHUNK):
            chunk = todo[i:i + CHUNK]
            for s, d in zip(chunk, run(chunk, tgt)):
                cache[key(s)] = d
        if todo:
            save_cache()
        return {"translations": [cache.get(key(t), t) for t in req.texts]}


### Cell 5: Write the Qwen3-4B Advisory server file to Drive


In [ ]:
%%writefile /content/drive/MyDrive/AdvisoryServer/advisory_server.py
import os, json, re, threading, copy
os.environ.setdefault("HF_HOME", "/content/hf_cache")

import torch
import requests
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# GPU present: Qwen3-4B in 4-bit through transformers.
# No GPU: a Qwen3 GGUF file through llama.cpp (choose the file with the two environment variables).
USE_GPU = bool(torch.cuda.is_available())
GPU_MODEL_ID = "Qwen/Qwen3-4B"
CPU_GGUF_REPO = os.environ.get("QWEN_GGUF_REPO", "Qwen/Qwen3-1.7B-GGUF")
CPU_GGUF_FILE = os.environ.get("QWEN_GGUF_FILE", "Qwen3-1.7B-Q8_0.gguf")
MODEL_ID = GPU_MODEL_ID if USE_GPU else f"{CPU_GGUF_REPO}/{CPU_GGUF_FILE}"
BACKEND = "transformers-4bit-gpu" if USE_GPU else "llama.cpp-cpu"
MAX_NEW_TOKENS = int(os.environ.get("ADVISORY_MAX_TOKENS", "300" if USE_GPU else "220"))
KB_PATH = os.environ.get("ADVISORY_KB_PATH", "/content/drive/MyDrive/SmartFarming/advisory_kb.json")
CACHE_FILE = os.environ.get("ADVISORY_CACHE_PATH", "/content/drive/MyDrive/advisory_cache.json")
TRANSLATE_URL = os.environ.get("TRANSLATE_URL", "http://127.0.0.1:8002/translate")

# False = the "treatment" text (products and doses) is copied from the knowledge base, never generated.
# True  = the model writes it, and any dose that is not in the knowledge base is replaced by the KB text.
LLM_WRITES_TREATMENT = False
CONF_GATE = 0.70
FIELDS = ["immediate_action", "treatment", "prevention", "monitoring"]

# Fixed texts: have a native speaker review the Hindi and Gujarati wording before release.
DISCLAIMERS = {
    "en": "DISCLAIMER: Confirm the diagnosis and any chemical use with your local KVK or agriculture officer, and follow the product label.",
    "hi": "अस्वीकरण: कोई भी रसायन या दवा इस्तेमाल करने से पहले अपने स्थानीय कृषि विज्ञान केंद्र (KVK) या कृषि अधिकारी से निदान की पुष्टि करें और उत्पाद के लेबल का पालन करें।",
    "gu": "અસ્વીકરણ: કોઈ પણ રસાયણ કે દવાનો ઉપયોગ કરતા પહેલાં તમારા સ્થાનિક કૃષિ વિજ્ઞાન કેન્દ્ર (KVK) અથવા ખેતીવાડી અધિકારી પાસેથી નિદાનની પુષ્ટિ કરો અને ઉત્પાદનના લેબલનું પાલન કરો.",
}

# ---- knowledge base ----
print(f"[Init] Loading knowledge base from {KB_PATH}...")
ADVISORY_KB = {}
if os.path.exists(KB_PATH):
    try:
        with open(KB_PATH, "r", encoding="utf-8") as f:
            ADVISORY_KB = json.load(f)
        print(f"[Init] Loaded {len(ADVISORY_KB)} crop-disease knowledge entries.")
    except Exception as e:
        print(f"[Init] WARNING: could not load knowledge base: {e}")
else:
    print("[Init] WARNING: knowledge base file not found. Every request will get a 'not found' reply.")

# ---- model ----
lock = threading.Lock()
if USE_GPU:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    print(f"[Init] Loading {GPU_MODEL_ID} in 4-bit NF4 on the GPU...")
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained(GPU_MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(GPU_MODEL_ID, quantization_config=bnb, device_map={"": 0}).eval()
else:
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    print(f"[Init] No GPU. Downloading {CPU_GGUF_REPO}/{CPU_GGUF_FILE} ...")
    gguf_path = hf_hub_download(CPU_GGUF_REPO, CPU_GGUF_FILE)
    print(f"[Init] Loading it with llama.cpp on {os.cpu_count()} CPU threads...")
    llm = Llama(model_path=gguf_path, n_ctx=2048, n_threads=os.cpu_count() or 2, verbose=False)

# ---- cache (English advisories only; translation is done per request) ----
cache = {}
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, encoding="utf-8") as f:
            cache = json.load(f)
        print(f"[Cache] Loaded {len(cache)} cached advisories.")
    except Exception as e:
        print(f"[Cache] Could not load cache: {e}")

def save_cache():
    try:
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False)
    except Exception:
        pass

# ---- helpers (pure functions, no model needed) ----
NATIVE_DIGITS = str.maketrans("०१२३४५६७८९૦૧૨૩૪૫૬૭૮૯", "01234567890123456789")
DOSE_RE = re.compile(
    r"\d+(?:\.\d+)?\s*(?:ml|gms?|g|kg|litres?|liters?|l|ppm)\b(?:\s*(?:/|per)\s*[a-z]+)?", re.I)

def norm(s):
    return re.sub(r"\s+", "", str(s).lower())

def kb_blob(kb):
    return norm(json.dumps(kb, ensure_ascii=False))

def dose_tokens(text):
    return [norm(m.group(0)) for m in DOSE_RE.finditer(text or "")]

def doses_ok(text, blob):
    """True when every dose-like quantity in the text also appears in the knowledge entry."""
    return all(t in blob for t in dose_tokens(text))

def as_list(x):
    if x is None or x == "":
        return []
    return x if isinstance(x, list) else [x]

def item_text(x):
    if isinstance(x, dict):
        parts = [f"{k.replace('_', ' ')}: {v}" for k, v in x.items() if k != "source" and v not in (None, "", [])]
        return ", ".join(parts)
    return str(x)

def kb_treatment(kb):
    org = [item_text(o) for o in as_list(kb.get("organic_options"))]
    chem = [item_text(c) for c in as_list(kb.get("chemical_options"))]
    parts = []
    if org:
        parts.append("Organic options: " + "; ".join(org) + ".")
    if chem:
        parts.append("Chemical options (follow the product label): " + "; ".join(chem) + ".")
    return " ".join(parts) or "No specific treatment is listed. Consult your local KVK or agriculture officer."

def fallback_fields(kb):
    cult = [item_text(c) for c in as_list(kb.get("cultural_practices"))]
    prec = [item_text(p) for p in as_list(kb.get("precautions"))]
    return {
        "immediate_action": cult[0] if cult else "Separate the affected plants and consult your local KVK.",
        "treatment": kb_treatment(kb),
        "prevention": " ".join(prec or cult) or "Keep the field clean and avoid overcrowding of plants.",
        "monitoring": "Check the field regularly and note whether the symptoms spread. Contact your local KVK if they do.",
    }

def split_sentences(text):
    return [p for p in re.split(r"(?<=[.!?])\s+", (text or "").strip()) if p]

def digits_of(text):
    return re.findall(r"\d+(?:\.\d+)?", (text or "").translate(NATIVE_DIGITS))

def translate_fields(data, lang):
    """Translate sentence by sentence. A field whose numbers change is kept in English."""
    sents, owner = [], []
    for k in FIELDS:
        for s in split_sentences(data[k]):
            sents.append(s)
            owner.append(k)
    if not sents:
        return {k: data[k] for k in FIELDS}, "ok"
    r = requests.post(TRANSLATE_URL, json={"texts": sents, "target": lang}, timeout=180)
    r.raise_for_status()
    out = r.json().get("translations", [])
    if len(out) != len(sents):
        raise ValueError("translation length mismatch")
    per = {k: [] for k in FIELDS}
    for k, dst in zip(owner, out):
        per[k].append(dst)
    result, kept_english = {}, []
    for k in FIELDS:
        joined = " ".join(per[k])
        if digits_of(joined) == digits_of(data[k]):
            result[k] = joined
        else:
            result[k] = data[k]
            kept_english.append(k)
    return result, ("partial" if kept_english else "ok")
# ---- end helpers ----

def _chatml(system_prompt, user_prompt):
    # Same layout as the Qwen3 chat template with thinking switched off (empty think block)
    return (f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
            "<|im_start|>assistant\n<think>\n\n</think>\n\n")

_GRAMMARS = {}
def _json_grammar(keys):
    """llama.cpp grammar that forces a JSON object with exactly these string keys (CPU path only)."""
    if not keys:
        return None
    key = tuple(keys)
    if key not in _GRAMMARS:
        try:
            from llama_cpp import LlamaGrammar
            schema = {"type": "object", "properties": {k: {"type": "string"} for k in keys}, "required": list(keys)}
            _GRAMMARS[key] = LlamaGrammar.from_json_schema(json.dumps(schema))
        except Exception as e:
            print(f"[Grammar] not available ({e}); continuing without it")
            _GRAMMARS[key] = None
    return _GRAMMARS[key]

def generate_llm(system_prompt, user_prompt, max_new_tokens=None, json_keys=None):
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
    if USE_GPU:
        msgs = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer([text], return_tensors="pt").to("cuda")
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=True, temperature=0.2, top_p=0.8, top_k=20)
        return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    kw = dict(max_tokens=max_new_tokens, temperature=0.2, top_p=0.8, top_k=20, stop=["<|im_end|>"])
    grammar = _json_grammar(json_keys)
    if grammar is not None:
        kw["grammar"] = grammar
    out = llm.create_completion(_chatml(system_prompt, user_prompt), **kw)
    return out["choices"][0]["text"].strip()

try:
    generate_llm("You are an agronomy assistant.", "Say Ready.", max_new_tokens=5)
    extra = f" VRAM allocated: {round(torch.cuda.memory_allocated() / 1e6)} MB" if USE_GPU else ""
    print(f"[Init] {MODEL_ID} warmed up ({BACKEND}).{extra}")
except Exception as e:
    print(f"[Init] Warmup notice: {e}")

def build_advisory(kb, req, crop, disease):
    """Returns English advisory fields plus sources. Model output is checked against the KB."""
    chem = as_list(kb.get("chemical_options"))
    org = as_list(kb.get("organic_options"))
    cult = [item_text(c) for c in as_list(kb.get("cultural_practices"))]
    blob = kb_blob(kb)
    keys = FIELDS if LLM_WRITES_TREATMENT else [k for k in FIELDS if k != "treatment"]
    desc = {
        "immediate_action": "One or two sentences: what to do today, considering the severity and weather",
        "treatment": "Treatment using ONLY the products and doses in the facts",
        "prevention": "Two or three sentences from the cultural practices and precautions",
        "monitoring": "What to check, and how often, over the next one to two weeks",
    }
    system_prompt = (
        "You are an agricultural advisory assistant for Indian farmers. "
        "Use ONLY the verified facts provided. Never name a chemical, product or dose that is not in the facts. "
        "Write short, simple sentences a farmer can follow, and keep every value under 40 words. "
        "Reply with one JSON object and nothing else."
    )
    user_prompt = f"""Verified facts:
- Crop: {crop}
- Diagnosis: {disease}
- Symptoms: {kb.get('symptoms')}
- Cultural practices: {', '.join(cult)}
- Organic options: {json.dumps(org, ensure_ascii=False)}
- Chemical options: {json.dumps(chem, ensure_ascii=False)}
- Precautions: {kb.get('precautions')}

Field situation:
- Severity: {req.severity_bucket} ({req.severity_pct}% of leaf area affected)
- Weather: {req.temperature} C, {req.weather_condition}, {req.humidity}% humidity
- Plot area: {req.plot_acres or 'standard field'} acres

Return ONLY a JSON object with exactly these keys:
{json.dumps({k: desc[k] for k in keys}, indent=2)}
"""
    with lock:
        raw = generate_llm(system_prompt, user_prompt, json_keys=keys)
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        parsed = json.loads(m.group(0)) if m else {}
    except Exception:
        parsed = {}
    if not isinstance(parsed, dict):
        parsed = {}

    fb = fallback_fields(kb)
    data = {}
    for k in FIELDS:
        v = parsed.get(k) if k in keys else None
        if isinstance(v, list):
            v = " ".join(str(x) for x in v)
        if not isinstance(v, str) or not v.strip() or not doses_ok(v, blob):
            v = fb[k]
        data[k] = v.strip()

    srcs = [i.get("source") for i in chem + org if isinstance(i, dict) and i.get("source")]
    srcs += [str(s) for s in as_list(kb.get("sources"))]
    data["sources"] = list(dict.fromkeys(str(s) for s in srcs)) or ["Curated knowledge base (advisory_kb.json)"]
    return data

def package(data, lang, **extra):
    out = {"language": "en"}
    out.update({k: data[k] for k in FIELDS})
    out["sources"] = data.get("sources", [])
    out.update(extra)
    status = "not_requested"
    if lang != "en":
        try:
            translated, status = translate_fields(data, lang)
            out["english"] = {k: data[k] for k in FIELDS}
            out.update(translated)
            out["language"] = lang
        except Exception as e:
            status = "unavailable"
            print(f"[Translate] failed: {e}")
    out["translation_status"] = status
    out["safety_disclaimer"] = DISCLAIMERS.get(out["language"], DISCLAIMERS["en"])
    return out

# ---- API ----
app = FastAPI(title="Qwen3-4B Crop Advisory Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class AdvisoryRequest(BaseModel):
    crop: str
    disease: str
    confidence: float
    severity_pct: float = 0.0
    severity_bucket: str = "Unknown"
    temperature: str = "N/A"
    humidity: str = "N/A"
    weather_condition: str = "N/A"
    plot_acres: float | None = None
    language: str = "en"   # "en", "hi" or "gu"

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "service": "qwen_advisory",
        "model": MODEL_ID,
        "backend": BACKEND,
        "classes_in_kb": len(ADVISORY_KB),
        "llm_writes_treatment": LLM_WRITES_TREATMENT,
        "vram_mb": round(torch.cuda.memory_allocated() / 1e6) if USE_GPU else None,
    }

@app.post("/advise")
def advise(req: AdvisoryRequest):
    lang = (req.language or "en").lower()
    if lang not in ("en", "hi", "gu"):
        lang = "en"
    crop, disease = req.crop.strip(), req.disease.strip()
    kb_key = f"{crop}|{disease}"
    conf = req.confidence / 100 if req.confidence > 1 else req.confidence   # accept 0-1 or 0-100
    is_healthy = disease.lower() == "healthy"

    # Stage 1: rule gate. Low confidence goes to an expert, not to the model.
    if conf < CONF_GATE and not is_healthy:
        data = {
            "immediate_action": f"The automatic diagnosis for {crop} is not confident enough ({conf * 100:.0f}%). Please show the plant to your local KVK or agriculture officer before spraying anything.",
            "treatment": "No chemical treatment is recommended until an expert confirms the diagnosis.",
            "prevention": "Keep the field well drained and separate suspicious plants.",
            "monitoring": "Watch whether the symptoms spread over the next 48 hours.",
            "sources": ["Local KVK / agriculture extension office"],
        }
        return package(data, lang, escalate_to_expert=True, reason="low_confidence")

    kb = ADVISORY_KB.get(kb_key)
    if not isinstance(kb, dict) or not kb:
        data = {
            "immediate_action": f"No verified guidance is stored for {crop} - {disease}. Please consult your local KVK or agriculture officer.",
            "treatment": "No chemical treatment is recommended without expert advice.",
            "prevention": "Keep the field clean and avoid overcrowding of plants.",
            "monitoring": "Check the plants regularly for spreading symptoms.",
            "sources": ["Local KVK / agriculture extension office"],
        }
        return package(data, lang, escalate_to_expert=True, reason="no_knowledge_entry")

    # Stage 2 and 3: constrained generation, checked against the knowledge entry
    cache_key = f"{kb_key}|{req.severity_bucket}|{req.weather_condition}"
    data = cache.get(cache_key)
    if data is None:
        data = build_advisory(kb, req, crop, disease)
        cache[cache_key] = data
        save_cache()
    return package(copy.deepcopy(data), lang)

### Cell 6: Start the servers in the background (8001, 8002, then 8003)

In [ ]:
import os, subprocess, time, signal, requests
from google.colab import userdata

HAS_GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
print("GPU detected: Qwen3-4B runs in 4-bit." if HAS_GPU else "No GPU: all three services run on CPU; Qwen uses a GGUF model through llama.cpp.")

# Only used when there is no GPU.  "1.7B" is faster (recommended on CPU); "4B" is better but about twice as slow.
QWEN_CPU_MODEL = "1.7B"
CPU_MODELS = {"1.7B": ("Qwen/Qwen3-1.7B-GGUF", "Qwen3-1.7B-Q8_0.gguf"),
              "4B": ("Qwen/Qwen3-4B-GGUF", "Qwen3-4B-Q4_K_M.gguf")}

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add HF_TOKEN in Colab Secrets (key icon) and enable Notebook access"
print("HF_TOKEN loaded")

def kill_matching(pattern):
    pids = subprocess.run(["pgrep", "-f", pattern], capture_output=True, text=True).stdout.split()
    for pid in pids:
        try:
            os.kill(int(pid), signal.SIGTERM)
        except Exception:
            pass

for pat in ["uvicorn main:app", "uvicorn translate_server:app", "uvicorn advisory_server:app", "cloudflared tunnel"]:
    kill_matching(pat)
time.sleep(2)

os.makedirs("/content/hf_cache", exist_ok=True)

def start(cmd, cwd, log):
    env = {**os.environ, "HF_TOKEN": HF_TOKEN, "HF_HOME": "/content/hf_cache",
           "QWEN_GGUF_REPO": CPU_MODELS[QWEN_CPU_MODEL][0], "QWEN_GGUF_FILE": CPU_MODELS[QWEN_CPU_MODEL][1]}
    return subprocess.Popen(cmd, cwd=cwd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

def tail(path, n=30):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

def wait_ready(name, url, proc, log, timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            print(f"{name}: process exited (code {proc.returncode}). Last log lines:\n{tail(log)}")
            return False
        try:
            if requests.get(url, timeout=2).status_code < 500:
                print(f"{name}: ready after {int(time.time() - t0)}s")
                return True
        except Exception:
            pass
        time.sleep(3)
    print(f"{name}: not ready after {timeout}s. Last log lines:\n{tail(log)}")
    return False

# 1 and 2: vision (8001) and translation (8002) start together
vision_proc = start(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8001"],
                    "/content/drive/MyDrive/SmartFarming", "/content/vision.log")
translate_proc = start(["uvicorn", "translate_server:app", "--host", "0.0.0.0", "--port", "8002"],
                       "/content/drive/MyDrive/IndicTransServer", "/content/translate.log")
vision_ok = wait_ready("vision", "http://127.0.0.1:8001/docs", vision_proc, "/content/vision.log")
translate_ok = wait_ready("translate", "http://127.0.0.1:8002/health", translate_proc, "/content/translate.log")

# 3: advisory (8003) starts only after the other two are up, so the three model loads
# do not compete for RAM (a free Colab runtime has about 12 GB)
advisory_proc = start(["uvicorn", "advisory_server:app", "--host", "0.0.0.0", "--port", "8003"],
                      "/content/drive/MyDrive/AdvisoryServer", "/content/advisory.log")
advisory_ok = wait_ready("advisory", "http://127.0.0.1:8003/health", advisory_proc,
                         "/content/advisory.log", timeout=1500)

### Cell 7: One Cloudflare tunnel per port (8001, 8002, 8003)


In [ ]:
import re, json, time, subprocess

def open_tunnel(port, timeout=60):
    log = f"/content/tunnel_{port}.log"
    proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
        stdout=open(log, "w"), stderr=subprocess.STDOUT)
    end = time.time() + timeout
    while time.time() < end:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open(log).read())
        if m:
            return proc, m.group(0)
        time.sleep(1)
    raise RuntimeError(f"No tunnel URL for port {port}. Check {log}")

vision_tunnel, VISION_URL = open_tunnel(8001)
translate_tunnel, TRANSLATE_URL = open_tunnel(8002)
ADVISORY_URL = None
if advisory_ok:
    advisory_tunnel, ADVISORY_URL = open_tunnel(8003)

print("==================================================")
print("VISION_SERVICE_URL     =", VISION_URL)
print("TRANSLATE_SERVICE_URL  =", TRANSLATE_URL)
print("ADVISORY_SERVICE_URL   =", ADVISORY_URL or "(not running: startup failed, see /content/advisory.log)")
print("==================================================")

json.dump({"vision": VISION_URL, "translate": TRANSLATE_URL, "advisory": ADVISORY_URL},
          open("/content/drive/MyDrive/service_urls.json", "w"))

### Cell 7b: Auto-sync Quick Tunnel URLs to Cloudflare Worker (Option B Gateway)

Synchronizes the three new `trycloudflare.com` URLs directly to your permanent Cloudflare Worker.

**Requirements for Zero-Touch Automation:**
- `CF_API_TOKEN`: Cloudflare API token with `Account.Workers Scripts: Edit` permissions (saved in Colab Secrets).
- `CF_ACCOUNT_ID`: Your Cloudflare Account ID (saved in Colab Secrets).
- `CF_WORKER_NAME`: Optional worker name (defaults to `smart-farming-gateway`).

*If Colab secrets are not configured, you can simply copy the URLs printed above into the Cloudflare Worker Settings > Variables tab.*

In [ ]:
#@title Auto-sync Quick Tunnel URLs to Cloudflare Worker Gateway
import requests
from google.colab import userdata

try:
    CF_API_TOKEN = userdata.get('CF_API_TOKEN')
    CF_ACCOUNT_ID = userdata.get('CF_ACCOUNT_ID')
    CF_WORKER_NAME = (userdata.get('CF_WORKER_NAME') or 'smart-farming-gateway').strip()

    if CF_API_TOKEN and CF_ACCOUNT_ID:
        print(f'[Cloudflare Gateway] Syncing tunnel URLs to Worker \'{CF_WORKER_NAME}\'...')
        cf_headers = {
            'Authorization': f'Bearer {CF_API_TOKEN}',
            'Content-Type': 'application/json'
        }
        tunnels = [
            ('VISION_URL', VISION_URL),
            ('TRANSLATION_URL', TRANSLATE_URL),
            ('ADVISORY_URL', ADVISORY_URL),
        ]
        all_synced = True
        for var_name, var_url in tunnels:
            if not var_url:
                continue
            url = f'https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/workers/scripts/{CF_WORKER_NAME}/secrets'
            payload = {'name': var_name, 'text': var_url, 'type': 'secret_text'}
            resp = requests.put(url, headers=cf_headers, json=payload)
            if resp.status_code == 200:
                print(f'  ✓ {var_name} -> {var_url}')
            else:
                all_synced = False
                print(f'  ✗ Failed to update {var_name}: HTTP {resp.status_code} - {resp.text}')
        if all_synced:
            print('\n[Cloudflare Gateway] SUCCESS: All 3 upstream services are now live through your Worker!')
    else:
        print('[Cloudflare Gateway] Note: CF_API_TOKEN or CF_ACCOUNT_ID not found in Colab Secrets.')
        print('[Cloudflare Gateway] Manual update: Copy the 3 URLs above into Cloudflare Worker Settings > Variables.')
except Exception as e:
    print(f'[Cloudflare Gateway] Note: Auto-sync skipped or encountered an error: {e}')


### Cell 8: Test all 3 public URLs


In [ ]:
import requests, time, json

def retry(fn, tries=20, delay=3):
    for _ in range(tries):
        try:
            r = fn()
            if r.status_code < 500:
                return r
        except Exception:
            pass
        time.sleep(delay)   # a new tunnel can take a little while before it resolves
    return None

# 1. Translation
r = retry(lambda: requests.get(f"{TRANSLATE_URL}/health", timeout=10))
print("translate health:", r.json() if r is not None else "FAILED")
for lang in ["hi", "gu"]:
    r = retry(lambda: requests.post(f"{TRANSLATE_URL}/translate",
                                    json={"texts": ["Your crop shows signs of leaf blight."], "target": lang}, timeout=120))
    print(f"translate {lang} ->", r.json() if r is not None else "FAILED")

# 2. Vision
r = retry(lambda: requests.get(f"{VISION_URL}/docs", timeout=10))
print("vision status:", r.status_code if r is not None else "FAILED")

# 3. Advisory. Calls go to localhost first so the real generation time is visible (a public Cloudflare
#    tunnel cuts a request after about 100 s). The same case is then repeated through the tunnel; it is
#    cached by then, so it answers fast and proves the tunnel works.
LOCAL_ADVISORY = "http://127.0.0.1:8003"
if ADVISORY_URL:
    r = retry(lambda: requests.get(f"{LOCAL_ADVISORY}/health", timeout=10))
    print("advisory health:", r.json() if r is not None else "FAILED")

    payload = {"crop": "Tomato", "disease": "Late Blight", "confidence": 0.92,
               "severity_pct": 24.5, "severity_bucket": "Moderate", "temperature": "22",
               "humidity": "85", "weather_condition": "Rainy", "plot_acres": 2.0}

    def call(label, base, body, timeout=600):
        t0 = time.time()
        try:
            r = requests.post(f"{base}/advise", json=body, timeout=timeout)
            print(f"\n/advise [{label}] HTTP {r.status_code} in {time.time() - t0:.1f}s")
            try:
                print(json.dumps(r.json(), indent=2, ensure_ascii=False))
            except Exception:
                print(r.text[:300])
        except Exception as e:
            print(f"\n/advise [{label}] FAILED after {time.time() - t0:.1f}s: {e}")

    call("local, English (first call generates)", LOCAL_ADVISORY, {**payload, "language": "en"})
    call("public tunnel, English (cached)", ADVISORY_URL, {**payload, "language": "en"}, timeout=120)
    call("local, Hindi (cached advisory + translation)", LOCAL_ADVISORY, {**payload, "language": "hi"})
    call("local, low-confidence gate", LOCAL_ADVISORY, {**payload, "confidence": 0.40, "language": "en"})
else:
    print("advisory: not running. Check /content/advisory.log (Cell 9).")

### Cell 9 (optional): Logs and GPU memory status


In [ ]:
for name in ["vision.log", "translate.log", "advisory.log", "tunnel_8001.log", "tunnel_8002.log", "tunnel_8003.log"]:
    print(f"\n===== {name} (last 20 lines) =====")
    !tail -20 /content/{name} 2>/dev/null || echo "Not found"

print("\nGPU status:")
!nvidia-smi || echo "No GPU on this runtime"